# AgriPrice-IQ: Step 1 - Data Cleaning & Ingestion Pipeline

This notebook processes the raw StatCan monthly retail price CSV files, transforms them from a wide format to a long time-series format, handles data cleaning (dates and suppressed values), maps food items into **7 commercial categories**, and loads the final master dataset into a local **SQLite database (`database/agriprice_iq.db`)**.

In [48]:
import pandas as pd
import glob
import os
import sqlite3

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [49]:
import glob
import os
import pandas as pd

path = "../data_provinces/"
all_files = glob.glob(os.path.join(path, "*.csv"))

print(f"Found {len(all_files)} provincial files:")

province_mapping = {
    'britishcolumbia': 'British Columbia',
    'Alberta': 'Alberta',
    'saskatchewan': 'Saskatchewan',
    'manitoba': 'Manitoba',
    'ontario': 'Ontario',
    'quebec': 'Quebec',
    'newbrunswick': 'New Brunswick',
    'novascotia': 'Nova Scotia',
    'princeedwardisland': 'Prince Edward Island',
    'newfoundlandlabrador': 'Newfoundland and Labrador'
}

provincial_dataframes = []

for f in all_files:
    file_name = os.path.basename(f)
    print(f" - Processing {file_name}")
    
    key = file_name.replace('.csv', '').lower().strip()
    
    province_name = province_mapping.get(key, key.replace('_', ' ').title())
    
    temp_df = pd.read_csv(f, skiprows=9, header=0, on_bad_lines='skip')
    
    temp_df = temp_df.iloc[1:].reset_index(drop=True)
    
    temp_df['province'] = province_name
    
    provincial_dataframes.append(temp_df)

raw_retail_prices_wide = pd.concat(provincial_dataframes, ignore_index=True)

print("\nCombined raw dataset shape:", raw_retail_prices_wide.shape)

print("Provinces loaded:", raw_retail_prices_wide['province'].unique())

Found 10 provincial files:
 - Processing BritishColumbia.csv
 - Processing NovaScotia.csv
 - Processing PrinceEdwardIsland.csv
 - Processing Quebec.csv
 - Processing NewfoundlandLabrador.csv
 - Processing Saskatchewan.csv
 - Processing Alberta.csv
 - Processing Ontario.csv
 - Processing Manitoba.csv
 - Processing NewBrunswick.csv

Combined raw dataset shape: (1212, 69)
Provinces loaded: ['British Columbia' 'Nova Scotia' 'Prince Edward Island' 'Quebec'
 'Newfoundland and Labrador' 'Saskatchewan' 'Alberta' 'Ontario' 'Manitoba'
 'New Brunswick']


### Step 2: Wide-to-Long Time-Series Transformation & Data Sanitization

In [50]:
# 1. Identify columns for melting
# 'Products' is the product name column from StatCan, 'province' is the column we added
id_cols = ['Products', 'province']
date_cols = [col for col in raw_retail_prices_wide.columns if col not in id_cols]

# 2. Melt from wide to long time-series format
melted_retail_df = pd.melt(
    raw_retail_prices_wide, 
    id_vars=id_cols, 
    value_vars=date_cols, 
    var_name='date_raw', 
    value_name='price'
)

# 3. Rename columns for clarity
melted_retail_df = melted_retail_df.rename(columns={
    'Products': 'product_name'
})

# 4. Clean dates and price values (handling commas and StatCan suppression flags like 'F')
melted_retail_df['date'] = pd.to_datetime(melted_retail_df['date_raw'], errors='coerce')
melted_retail_df['price'] = pd.to_numeric(
    melted_retail_df['price'].astype(str).str.replace(',', '').str.replace('F', ''), 
    errors='coerce'
)

# 5. Drop rows with missing prices or dates
clean_retail_prices_canada = melted_retail_df.dropna(subset=['price', 'date'])
print("Cleaned long dataset shape:", clean_retail_prices_canada.shape)
clean_retail_prices_canada.head(3)

Cleaned long dataset shape: (74369, 5)


/var/folders/zv/ljgjz5z10y93dzq0hqkt_5zh0000gn/T/ipykernel_5505/845097444.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  melted_retail_df['date'] = pd.to_datetime(melted_retail_df['date_raw'], errors='coerce')


,product_name,province,date_raw,price,date
0,"Beef stewing cuts, per kilogram 5",British Columbia,January 2021,15.22,2021-01-01
1,"Beef striploin cuts, per kilogram 5",British Columbia,January 2021,25.58,2021-01-01
2,"Beef top sirloin cuts, per kilogram 5",British Columbia,January 2021,17.49,2021-01-01


### Step 3: Mapping to 7 Commercial Categories & Database Export

In this step, we map raw StatCan product descriptions into our **7 core commercial retail categories** (e.g., *Dairy & Eggs*, *Meat & Poultry*, *Fresh Produce*), aggregate the monthly average prices, and export the clean master dataset into our local SQLite database (`database/agriprice_iq.db`).

In [51]:
def map_to_category(product_name):
    product_name = str(product_name).lower()
    
    # 1. Dairy & Eggs (includes margarine, plant milks, butter, cheese, eggs)
    if any(kw in product_name for kw in ['milk', 'cheese', 'butter', 'margarine', 'eggs', 'yogurt', 'cream']): 
        return 'Dairy & Eggs'
        
    # 2. Meat, Poultry & Seafood (includes beef, pork, chicken, bacon, wieners, salmon, shrimp, tuna)
    elif any(kw in product_name for kw in ['beef', 'chicken', 'pork', 'meat', 'sausage', 'turkey', 'bacon', 'wieners', 'salmon', 'shrimp', 'tuna', 'fish']): 
        return 'Meat & Poultry'
        
    # 3. Fresh Produce & Frozen Veg/Fruit (includes all fruits, vegetables, spinach, corn, peas, beans, berries)
    elif any(kw in product_name for kw in [
        'apple', 'banana', 'orange', 'pear', 'lemon', 'lime', 'grape', 'cantaloupe', 
        'strawberry', 'avocado', 'potato', 'tomato', 'cabbage', 'carrot', 'onion', 
        'celery', 'cucumber', 'mushroom', 'lettuce', 'broccoli', 'pepper', 'squash', 
        'salad', 'spinach', 'corn', 'peas', 'beans', 'berries', 'fruit', 'vegetables'
    ]): 
        return 'Fresh Produce'
        
    # 4. Bakery & Grains (includes bread, flatbread, crackers, cookies, pasta, rice, cereal, flour)
    elif any(kw in product_name for kw in ['bread', 'flatbread', 'pita', 'cracker', 'cookie', 'biscuit', 'flour', 'cereal', 'pasta', 'rice', 'bakery', 'pizza']): 
        return 'Bakery & Grains'
        
    # 5. Pantry Staples (includes sugar, canned goods, oils, sauces, peanut butter, mayo, lentils, tofu, hummus, nuts, seeds, baby food)
    elif any(kw in product_name for kw in [
        'sugar', 'soup', 'canned', 'oil', 'sauce', 'condiment', 'pantry', 
        'peanut butter', 'mayonnaise', 'lentils', 'tofu', 'hummus', 'salsa', 
        'dressing', 'almonds', 'peanuts', 'seeds', 'nuts', 'baby food', 'infant formula'
    ]): 
        return 'Pantry Staples'
        
    # 6. Beverages (includes coffee, tea, juices)
    elif any(kw in product_name for kw in ['coffee', 'tea', 'juice', 'drink', 'water', 'beverage', 'ketchup']): 
        return 'Beverages'
        
    # 7. Personal Care & Household (includes deodorant, toothpaste, shampoo, laundry detergent)
    elif any(kw in product_name for kw in ['deodorant', 'toothpaste', 'shampoo', 'detergent', 'soap', 'paper', 'personal care', 'household']): 
        return 'Personal Care & Household'
        
    else: 
        return 'Other / Unassigned'

# Apply category mapping
clean_retail_prices_canada['category'] = clean_retail_prices_canada['product_name'].apply(map_to_category)

# Quick check: Verify if any items fell into 'Other / Unassigned'
unassigned_check = clean_retail_prices_canada[clean_retail_prices_canada['category'] == 'Other / Unassigned']['product_name'].unique()
print(f"Items left in 'Other / Unassigned': {len(unassigned_check)}")
if len(unassigned_check) > 0:
    print(unassigned_check)
else:
    print("Success! 100% of your products have been successfully mapped into your 7 categories.")

Items left in 'Other / Unassigned': 0
Success! 100% of your products have been successfully mapped into your 7 categories.


/var/folders/zv/ljgjz5z10y93dzq0hqkt_5zh0000gn/T/ipykernel_5505/730298200.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_retail_prices_canada['category'] = clean_retail_prices_canada['product_name'].apply(map_to_category)


In [52]:
clean_retail_prices_canada.head()

,product_name,province,date_raw,price,date,category
0,"Beef stewing cuts, per kilogram 5",British Columbia,January 2021,15.22,2021-01-01,Meat & Poultry
1,"Beef striploin cuts, per kilogram 5",British Columbia,January 2021,25.58,2021-01-01,Meat & Poultry
2,"Beef top sirloin cuts, per kilogram 5",British Columbia,January 2021,17.49,2021-01-01,Meat & Poultry
3,"Beef rib cuts, per kilogram 5",British Columbia,January 2021,25.46,2021-01-01,Meat & Poultry
4,"Ground beef, per kilogram 5",British Columbia,January 2021,9.75,2021-01-01,Meat & Poultry


In [53]:
# Aggregate monthly average by date, province, and category
monthly_category_prices = clean_retail_prices_canada.groupby(
    ['date', 'province', 'category'], 
    as_index=False
)['price'].mean()

print("Final aggregated dataset shape:", monthly_category_prices.shape)

Final aggregated dataset shape: (4690, 4)


In [54]:
monthly_category_prices.head()

,date,province,category,price
0,2021-01-01,Alberta,Bakery & Grains,3.845455
1,2021-01-01,Alberta,Beverages,4.143333
2,2021-01-01,Alberta,Dairy & Eggs,4.170000
3,2021-01-01,Alberta,Fresh Produce,3.371333
4,2021-01-01,Alberta,Meat & Poultry,11.792333


In [57]:
# Ensure database directory exists and write to SQLite
os.makedirs('../database', exist_ok=True)
conn = sqlite3.connect('../database/agriprice_iq.db')
monthly_category_prices.to_sql('monthly_prices', conn, if_exists='replace', index=False)
clean_retail_prices_canada.to_sql('retail_prices', conn, if_exists='replace', index=False)

# Verify table contents
verify_df = pd.read_sql("SELECT province, category, COUNT(*) as record_count FROM monthly_prices GROUP BY province, category LIMIT 5", conn)
print("\nData successfully stored and verified in SQLite database!")
display(verify_df)

conn.close()


Data successfully stored and verified in SQLite database!


,province,category,record_count
0,Alberta,Bakery & Grains,67
1,Alberta,Beverages,67
2,Alberta,Dairy & Eggs,67
3,Alberta,Fresh Produce,67
4,Alberta,Meat & Poultry,67


In [56]:
# Export DataFrame clean_retail_prices_canada ke format CSV
output_path = '../data/clean_retail_prices_canada.csv'
clean_retail_prices_canada.to_csv(output_path, index=False)

print(f"File CSV berhasil disimpan di: {output_path}")

File CSV berhasil disimpan di: ../data/clean_retail_prices_canada.csv
